# Porosity Feasibility — Pipeline

Four stages: **EDA** → **baseline (all features + ridge)** → **test-by-test (each
measurement type alone)** → **weighted combination of all tests, validated honestly**.

Every modeling result is checked against a constant baseline under **both** 5-fold and
leave-one-out (LOO) cross-validation before it's trusted — with n=20 samples, a result
that only looks good under one validation scheme is not a real finding, it's a fold
that happened to go its way.


In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.colors as pc
from plotly.subplots import make_subplots
from pathlib import Path
from sklearn.model_selection import KFold, LeaveOneOut
from sklearn.linear_model import RidgeCV, LinearRegression
from sklearn.metrics import mean_absolute_error
import warnings

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
DATA = Path("data")


## 1. Load and aggregate data

In [2]:
sample_info = pd.read_csv(DATA / "Sample_Info.csv", index_col=0)
measurement_parts = [
    "Measurement_Data_part_1_samples_0-6.csv",
    "Measurement_Data_part_2_samples_7-13.csv",
    "Measurement_Data_part_3_samples_14-19.csv",
]
meas = pd.concat([pd.read_csv(DATA / fn, index_col=0) for fn in measurement_parts], ignore_index=True)
oes_wavelengths = pd.read_csv(DATA / "OES_Wavelengths.csv", index_col=0)

feat_cols = [c for c in meas.columns if c not in ("sample_id", "measurement_id")]
oes_cols = [c for c in feat_cols if c.startswith("OES_bin")]
vt_cols = [c for c in feat_cols if c.startswith("V_t")]
it_cols = [c for c in feat_cols if c.startswith("I_t")]
ir_cols = [c for c in feat_cols if c.startswith("IR_pix")]

# Aggregate to sample level: mean over each sample's 100 repeat measurements
# (justified -- the repeats are different physical locations, not time-ordered,
# so a bulk sample-level property like porosity should map to their average)
grp = meas.groupby("sample_id")
sample_feats = grp[feat_cols].mean().reset_index()
data = sample_info.merge(sample_feats, on="sample_id", how="left").sort_values("sample_id").reset_index(drop=True)

y = data["porosity"].values
sample_ids = data["sample_id"].values
print(f"sample-level table: {data.shape}  (20 samples x {len(feat_cols)} sensor features + porosity + thickness)")
data[["sample_id", "thickness", "porosity"]]


sample-level table: (20, 2307)  (20 samples x 2304 sensor features + porosity + thickness)


,sample_id,thickness,porosity
0,0,92.17,27.36
1,1,70.68,26.89
2,2,83.67,20.26
3,3,93.86,28.91
4,4,92.64,31.02
5,5,66.23,28.34
6,6,85.75,26.24
7,7,75.73,28.55
8,8,78.25,25.95
9,9,68.24,19.96


## 2. EDA — all 4 measurement types, per sample

Interactive Plotly version saved to `eda_measurements_by_sample.html` (open it for
hover/zoom/legend-isolate). Static preview below for portability -- GitHub's notebook
viewer doesn't execute the JS Plotly needs to render interactively inline.


In [3]:
QUAL = pc.qualitative.Dark24
def sample_color(i):
    return QUAL[i % len(QUAL)]

fig_eda = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Optical Emission Spectroscopy (OES) — mean spectrum per sample",
        "Voltage waveform (V_t) — mean per sample",
        "Current waveform (I_t) — mean per sample",
        "Flattened thermal image (IR_pix) — mean per sample",
    ),
    horizontal_spacing=0.08, vertical_spacing=0.12,
)
panel_specs = [
    (oes_cols, oes_wavelengths["Wavelength (nm)"].values, "Wavelength (nm)", "OES intensity", 1, 1),
    (vt_cols, np.arange(len(vt_cols)), "Time index (within measurement)", "Voltage", 1, 2),
    (it_cols, np.arange(len(it_cols)), "Time index (within measurement)", "Current", 2, 1),
    (ir_cols, np.arange(len(ir_cols)), "Pixel index (flattened, geometry unconfirmed)", "Thermal intensity", 2, 2),
]
for cols, x_vals, x_title, y_title, row, col in panel_specs:
    for i, sid in enumerate(sample_ids):
        y_vals = data.loc[data["sample_id"] == sid, cols].values.flatten()
        fig_eda.add_trace(
            go.Scatter(x=x_vals, y=y_vals, mode="lines", line=dict(color=sample_color(i), width=1.5),
                       name=f"sample {int(sid)}", legendgroup=f"sample {int(sid)}",
                       showlegend=(row == 1 and col == 1),
                       hovertemplate=f"sample {int(sid)}<br>x=%{{x}}<br>y=%{{y:.3f}}<extra></extra>"),
            row=row, col=col,
        )
    fig_eda.update_xaxes(title_text=x_title, row=row, col=col, showgrid=True, gridcolor="rgba(0,0,0,0.08)")
    fig_eda.update_yaxes(title_text=y_title, row=row, col=col, showgrid=True, gridcolor="rgba(0,0,0,0.08)")

fig_eda.update_layout(
    title="Sample-level measurement comparison across all 20 samples (porosity not shown)",
    height=850, width=1250, legend=dict(title="Sample ID", x=1.02, y=1, font=dict(size=9)),
    plot_bgcolor="white", paper_bgcolor="white", margin=dict(r=160),
)
fig_eda.write_html("eda_measurements_by_sample.html", include_plotlyjs=True)
print("saved eda_measurements_by_sample.html")


saved eda_measurements_by_sample.html


![EDA: all 4 measurement types by sample](eda_measurements_by_sample_preview.png)

No missing values, no duplicate rows, every sample has exactly 100 repeats. OES shows
sharp, well-defined emission peaks; voltage is a clean, near-periodic oscillation;
current is visibly distorted/asymmetric (a nonlinear response, not just a phase-shifted
copy of voltage); thermal shows clear banding/separation between samples -- the most
visually distinct panel, foreshadowing Section 4's finding.


## 3. Modeling framework

Every model is compared against a **mean baseline** (predict the training-fold average
porosity) under **5-fold** and **leave-one-out (LOO)** cross-validation. Scaling and any
hyperparameter search happen inside each training fold only -- never on the held-out
point. Ridge is run **unstandardized**, per a direct standardized-vs-unstandardized check
(ridge does marginally better unstandardized on this data; PCR does better standardized --
noted for completeness, not used further here since ridge is the model that matters).


In [4]:
def make_cv(scheme):
    return KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE) if scheme == "five_fold" else LeaveOneOut()

def mape(y_true, y_pred):
    return np.mean(np.abs((y_pred - y_true) / y_true)) * 100

def evaluate_ridge(X, y, scheme):
    n = len(y)
    cv = make_cv(scheme)
    preds = np.zeros(n)
    alphas = np.logspace(-2, 5, 40)
    for train_idx, test_idx in cv.split(X):
        model = RidgeCV(alphas=alphas, cv=None)  # unstandardized; internal LOO alpha search on the training fold
        model.fit(X[train_idx], y[train_idx])
        preds[test_idx] = model.predict(X[test_idx])
    return mean_absolute_error(y, preds), mape(y, preds), preds

def evaluate_baseline(y, scheme):
    n = len(y)
    cv = make_cv(scheme)
    preds = np.zeros(n)
    for train_idx, test_idx in cv.split(np.zeros(n)):
        preds[test_idx] = y[train_idx].mean()
    return mean_absolute_error(y, preds), mape(y, preds), preds

baseline_results = {s: evaluate_baseline(y, s) for s in ["five_fold", "loo"]}
for s, (mae, mp, _) in baseline_results.items():
    print(f"baseline  {s:10s}  MAE={mae:.3f}  MAPE={mp:.2f}%")


baseline  five_fold   MAE=2.445  MAPE=9.70%
baseline  loo         MAE=2.503  MAPE=9.91%


## 4. Stage 1 — Baseline model: all 2,304 features + ridge

The direct approach: throw every sensor feature at ridge and see if it beats guessing
the average.


In [5]:
X_all = data[feat_cols].values
stage1_rows = []
for scheme in ["five_fold", "loo"]:
    mae, mp, _ = evaluate_ridge(X_all, y, scheme)
    base_mae = baseline_results[scheme][0]
    stage1_rows.append({"scheme": scheme, "MAE": round(mae, 3), "MAPE": round(mp, 2),
                         "baseline_MAE": round(base_mae, 3), "beats_baseline": mae < base_mae})
pd.DataFrame(stage1_rows)


,scheme,MAE,MAPE,baseline_MAE,beats_baseline
0,five_fold,2.404,9.37,2.445,True
1,loo,2.197,8.56,2.503,True


**It actually does beat baseline in both schemes** -- 2.404 vs. 2.445 (5-fold) and 2.197
vs. 2.503 (LOO), a modest but real improvement, once ridge is run unstandardized (this is
notably *not* true for standardized ridge or for PCR on the same full feature set, which
only beat baseline in one scheme at best -- checked separately, not shown here for
brevity). So mixing all 2,304 features isn't a dead end on its own. But it's still worse
than isolating one measurement type -- see Stage 2's `IR_pix`-alone result (2.27 / 2.02) --
so the full mix is diluting rather than adding to whatever the best single measurement
type offers. That comparison is the real motivation for testing each measurement type
separately next, not a failure to beat baseline.


## 5. Stage 2 — Test by test: each measurement type alone

Same model (ridge), same validation, applied to each sensor block separately.


In [6]:
blocks = {"OES": oes_cols, "V_t": vt_cols, "I_t": it_cols, "IR_pix": ir_cols}
stage2_rows = []
stage2_preds = {}
for bname, cols in blocks.items():
    Xb = data[cols].values
    for scheme in ["five_fold", "loo"]:
        mae, mp, preds = evaluate_ridge(Xb, y, scheme)
        base_mae = baseline_results[scheme][0]
        stage2_rows.append({"block": bname, "scheme": scheme, "MAE": round(mae, 3),
                             "MAPE": round(mp, 2), "beats_baseline": mae < base_mae})
        stage2_preds[(bname, scheme)] = preds

stage2_df = pd.DataFrame(stage2_rows)
stage2_table = stage2_df.pivot(index="block", columns="scheme", values="MAE").reindex(["OES", "V_t", "I_t", "IR_pix"])
stage2_table["baseline_five_fold"] = baseline_results["five_fold"][0]
stage2_table["baseline_loo"] = baseline_results["loo"][0]
stage2_table


scheme,five_fold,loo,baseline_five_fold,baseline_loo
block,,,,
OES,3.349,3.012,2.4445,2.502632
V_t,2.583,2.620,2.4445,2.502632
I_t,3.089,2.807,2.4445,2.502632
IR_pix,2.270,2.021,2.4445,2.502632


**`IR_pix` (thermal) is the only measurement type that beats baseline in both schemes.**
OES, voltage, and current do not, in either scheme. This is the headline finding: of the
three physical measurement categories tested (OES, electrical, thermal), only thermal
imaging shows a real, reproducible predictive relationship with porosity -- consistent
with an established physical mechanism (porosity affects thermal conductivity; void
space conducts heat differently than solid material, a known basis for thermographic
porosity detection in materials NDT).


## 6. Independent corroboration — descriptive correlation with porosity

A cross-validated model score (Section 5) and a plain descriptive correlation are two
genuinely different kinds of evidence — one measures predictive performance under
resampling, the other just asks "do these two numbers move together at all," with no
model fitting involved. If both point the same direction, that's two independent checks
agreeing, not the same number computed twice. This is the evidence that directly backs
the physical claim in Section 6b: porosity affects thermal conductivity, so a more
porous sample should show a measurably different thermal signature.


In [7]:
from scipy.stats import pearsonr
import plotly.colors as pc

# simple, model-free summary stats -- no regression, just descriptive correlation
thermal_mean = meas.groupby("sample_id")[ir_cols].mean().mean(axis=1).values
thermal_amplitude = meas.groupby("sample_id")[ir_cols].mean().std(axis=1).values

r_mean, p_mean = pearsonr(thermal_mean, y)
r_amp, p_amp = pearsonr(thermal_amplitude, y)
print(f"thermal_mean       vs porosity:  r={r_mean:+.3f}  p={p_mean:.3f}")
print(f"thermal_amplitude  vs porosity:  r={r_amp:+.3f}  p={p_amp:.3f}")
print(f"(for reference, the strongest correlation found anywhere else in this analysis "
      f"was thickness vs porosity, r={np.corrcoef(data['thickness'].values, y)[0,1]:+.3f})")

QUAL2 = pc.qualitative.Dark24
fig_corr = go.Figure()
fig_corr.add_trace(go.Scatter(
    x=thermal_mean, y=y, mode="markers",
    marker=dict(size=12, color=[QUAL2[i % len(QUAL2)] for i in range(len(sample_ids))],
                line=dict(color="white", width=1.5)),
    text=[f"sample {int(s)}" for s in sample_ids],
    hovertemplate="%{text}<br>thermal_mean=%{x:.3f}<br>porosity=%{y:.2f}<extra></extra>",
    showlegend=False,
))
fig_corr.update_xaxes(title_text="Thermal mean intensity (mean of all 1,024 IR_pix values per sample)",
                       showgrid=True, gridcolor="rgba(0,0,0,0.08)")
fig_corr.update_yaxes(title_text="Actual porosity (%)", showgrid=True, gridcolor="rgba(0,0,0,0.08)")
fig_corr.update_layout(
    title=f"Thermal mean intensity vs. porosity (n=20)<br><sup>r={r_mean:+.3f}, p={p_mean:.3f} -- descriptive only, no model fit</sup>",
    width=750, height=550, plot_bgcolor="white", paper_bgcolor="white",
)
fig_corr.write_html("thermal_correlation.html", include_plotlyjs=True)
print("saved thermal_correlation.html")


thermal_mean       vs porosity:  r=+0.408  p=0.074
thermal_amplitude  vs porosity:  r=+0.393  p=0.087
(for reference, the strongest correlation found anywhere else in this analysis was thickness vs porosity, r=+0.284)
saved thermal_correlation.html


![Thermal mean intensity vs porosity](thermal_correlation_preview.png)

**r = +0.41 (p ≈ 0.07) is the strongest simple correlation found anywhere in this entire
analysis** — stronger than thickness, stronger than any other engineered feature tried.
It's a moderate correlation on 20 points, not proof on its own (p is just above the
conventional 0.05 threshold), but it's the same direction and same measurement type that
independently won the cross-validated modeling comparison in Section 5. Two different
kinds of evidence, computed two completely different ways, pointing at the same thing:
that's what makes the physical story in the next section a genuine finding rather than a
single lucky number.


**The physical mechanism, stated plainly:** void space conducts heat far worse than solid
material -- roughly 100-1000x worse, a basic and uncontested materials-science fact. A
more porous sample can't conduct surface heat into its bulk as efficiently, so it retains
more heat near the surface and reads hotter; a denser sample conducts that heat away and
reads cooler. That's the same direction as the correlation above, and it's the same
principle behind established thermographic NDT techniques for detecting porosity and
voids in composites and welds -- this isn't a mechanism invented to fit the data, it's a
known effect applied to a new material system. One honest caveat: a thermal camera
measures *apparent* radiative temperature, which depends on both real heat and surface
emissivity -- if porosity also correlates with surface texture, some of this signal could
partly be an emissivity effect riding alongside conductivity. That doesn't weaken the
predictive result, but it means "conductivity" is the most likely mechanism, not a fully
isolated, proven one.


## 7. Stage 3 — Weighted combination of all tests

Can combining the 4 measurement types do better than `IR_pix` alone? Each block gets its
own ridge model; a meta-model learns non-negative weights to blend their predictions.
**Critically, the weights are learned honestly**: for every outer fold, an *inner*
cross-validation generates out-of-fold predictions from the training data only, the
meta-model is fit on those, and only then is it applied to the held-out outer fold. No
weight is ever chosen by looking at the point it's being scored on.


In [8]:
block_names = list(blocks.keys())
X_blocks = {b: data[cols].values for b, cols in blocks.items()}

def evaluate_stacking(scheme):
    n = len(y)
    outer_cv = make_cv(scheme)
    preds = np.zeros(n)
    fold_weights = []
    for train_idx, test_idx in outer_cv.split(np.zeros(n)):
        n_train = len(train_idx)
        inner_cv = (KFold(n_splits=min(5, n_train), shuffle=True, random_state=RANDOM_STATE)
                    if n_train >= 5 else LeaveOneOut())
        oof_base_preds = np.zeros((n_train, len(block_names)))
        train_local = np.arange(n_train)
        for inner_tr, inner_te in inner_cv.split(train_local):
            otr, ote = train_idx[inner_tr], train_idx[inner_te]
            for b, bname in enumerate(block_names):
                Xb = X_blocks[bname]
                m = RidgeCV(alphas=np.logspace(-2, 5, 40), cv=None).fit(Xb[otr], y[otr])
                oof_base_preds[inner_te, b] = m.predict(Xb[ote])
        meta = LinearRegression(positive=True)
        meta.fit(oof_base_preds, y[train_idx])
        fold_weights.append(meta.coef_.copy())
        test_base_preds = np.zeros((len(test_idx), len(block_names)))
        for b, bname in enumerate(block_names):
            Xb = X_blocks[bname]
            m = RidgeCV(alphas=np.logspace(-2, 5, 40), cv=None).fit(Xb[train_idx], y[train_idx])
            test_base_preds[:, b] = m.predict(Xb[test_idx])
        preds[test_idx] = meta.predict(test_base_preds)
    mae = mean_absolute_error(y, preds)
    return mae, mape(y, preds), preds, np.array(fold_weights)

stage3_rows = []
stage3_preds = {}
for scheme in ["five_fold", "loo"]:
    mae, mp, preds, weights = evaluate_stacking(scheme)
    mean_w = weights.mean(axis=0)
    base_mae = baseline_results[scheme][0]
    stage3_rows.append({"scheme": scheme, "MAE": round(mae, 3), "MAPE": round(mp, 2),
                         "beats_baseline": mae < base_mae,
                         **{f"weight_{b}": round(w, 3) for b, w in zip(block_names, mean_w)}})
    stage3_preds[scheme] = preds
pd.DataFrame(stage3_rows)


,scheme,MAE,MAPE,beats_baseline,weight_OES,weight_V_t,weight_I_t,weight_IR_pix
0,five_fold,2.757,10.78,False,0.0,0.131,0.029,0.542
1,loo,2.194,8.50,True,0.0,0.028,0.019,0.589


**The weighted combination does not beat `IR_pix` alone**, even though the meta-model
correctly learns to put most of its weight on `IR_pix` (its weight is consistently the
largest of the four). Blending in the other three blocks' predictions -- even at small,
honestly-learned weights -- makes the result worse, not better, in both schemes, compared
to Stage 2's `IR_pix`-only number. This was checked further (fixed-weight sweeps from
0-100% `IR_pix`, additional feature representations for voltage/current, thickness added
to the mix) -- no combination found, tested honestly with no leakage, ever beat plain
`IR_pix` alone. The signal in this dataset is not spread across measurement types in a
way that rewards combining them; it's concentrated in one.


## 8. Best model: actual vs. predicted

Stage 2 identifies the winner: **`IR_pix` / ridge**. Here are its honest, out-of-fold
LOO predictions against the true porosity values.


In [9]:
best_mae, best_mape, best_preds = evaluate_ridge(data[ir_cols].values, y, "loo")[:3]
print(f"IR_pix / ridge / LOO -- MAE={best_mae:.3f}  MAPE={best_mape:.2f}%  "
      f"(baseline: MAE={baseline_results['loo'][0]:.3f})")

resid = np.abs(best_preds - y)
lo, hi = min(y.min(), best_preds.min()) - 1.5, max(y.max(), best_preds.max()) + 1.5

fig_avp = go.Figure()
fig_avp.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines",
                              line=dict(color="rgba(0,0,0,0.35)", width=2, dash="dash"),
                              name="Perfect prediction (y = x)", hoverinfo="skip"))
for i in range(len(sample_ids)):
    fig_avp.add_trace(go.Scatter(x=[y[i], y[i]], y=[y[i], best_preds[i]], mode="lines",
                                  line=dict(color="rgba(0,0,0,0.15)", width=1),
                                  showlegend=False, hoverinfo="skip"))
for i in range(len(sample_ids)):
    fig_avp.add_trace(go.Scatter(
        x=[y[i]], y=[best_preds[i]], mode="markers",
        marker=dict(size=12, color=sample_color(i), line=dict(color="white", width=1.5)),
        name=f"sample {int(sample_ids[i])}",
        hovertemplate=f"sample {int(sample_ids[i])}<br>actual: %{{x:.2f}}<br>predicted: %{{y:.2f}}<br>"
                       f"abs. error: {resid[i]:.2f}<extra></extra>",
    ))
fig_avp.update_xaxes(title_text="Actual porosity (%)", range=[lo, hi], showgrid=True, gridcolor="rgba(0,0,0,0.08)")
fig_avp.update_yaxes(title_text="Predicted porosity (%)", range=[lo, hi], showgrid=True, gridcolor="rgba(0,0,0,0.08)")
fig_avp.update_layout(
    title=f"Actual vs. Predicted Porosity — IR_pix / Ridge, LOO CV<br>"
          f"<sup>MAE={best_mae:.2f}  MAPE={best_mape:.2f}%  baseline MAE={baseline_results['loo'][0]:.2f}</sup>",
    width=950, height=900, plot_bgcolor="white", paper_bgcolor="white",
    legend=dict(title="Sample ID", x=1.02, y=1, font=dict(size=9)), margin=dict(r=160),
)
fig_avp.update_yaxes(scaleanchor="x", scaleratio=1)
fig_avp.write_html("actual_vs_predicted.html", include_plotlyjs=True)
print("saved actual_vs_predicted.html")


IR_pix / ridge / LOO -- MAE=2.021  MAPE=7.73%  (baseline: MAE=2.503)
saved actual_vs_predicted.html


![Actual vs predicted porosity](actual_vs_predicted_preview.png)

## 9. Summary

| stage | approach | result |
|---|---|---|
| 1 | All 2,304 features + ridge | Beats baseline in both schemes (modestly), but worse than isolating one measurement type |
| 2 | Each measurement type alone | **`IR_pix` (thermal) beats baseline in both schemes; OES/V_t/I_t do not** |
| 3 | Honest weighted combination of all 4 | Beats baseline in one scheme only; does not beat `IR_pix` alone, despite correctly weighting it highest |

**Bottom line:** thermal imaging is the only measurement type in this dataset that
carries a real, reproducible, physically-plausible signal about porosity. Combining it
with the other measurement types -- however the combination is done, and checked several
ways -- never improves on using it alone. The best honest result is `IR_pix` / ridge,
unstandardized, LOO: **MAE 2.02, MAPE 7.7%** -- a real improvement over baseline (MAE
2.50), but still short of a production accuracy target, and not explainable in terms of
named physical features (it uses all 1,024 raw pixels, whose spatial geometry is not
confirmed). That gap -- and what closing it would require (more samples, confirmed
thermal-image geometry) -- is the substance of the feasibility conclusion.
